In [ ]:
%pip install -U agentops

In [1]:
import agentops
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
import os
import re


from typing import TypedDict, List, Dict, Any


In [2]:
import os
from dotenv import load_dotenv

# Carregar o arquivo .env
load_dotenv(
    r"C:\Users\eugen\Documents\ProjetoIA\.env",
    override=True
)

# Verificar configurações
print("OpenAI:", bool(os.getenv("OPENAI_API_KEY")))
print("Tavily:", bool(os.getenv("TAVILY_API_KEY")))
print("AgentOps:", bool(os.getenv("AGENTOPS_API_KEY")))

OpenAI: True
Tavily: True
AgentOps: True


In [ ]:
import agentops
from dotenv import load_dotenv

# Carregar variáveis de ambiente
load_dotenv(
    r"C:\Users\eugen\Documents\ProjetoIA\.env",
    override=True
)
agentops.init(
    api_key=os.environ["AGENTOPS_API_KEY"],
    default_tags=["langgraph-agent"],
    auto_start_session=False
)

In [4]:
class AgentState(TypedDict):
    query: str
    search_results: str
    summary: str
    final_answer: str

In [5]:
# Inicializar o modelo da OpenAI
model = ChatOpenAI(model_name="gpt-4o", temperature=0)

# Inicializar o Tavily Search
from langchain_tavily import TavilySearch

search = TavilySearch(max_results=3)

In [6]:

from langchain_tavily import TavilySearch
from agentops.sdk.decorators import tool as agentops_tool

@agentops_tool(name="web_search")
def web_search(state: AgentState) -> AgentState:
    """Realiza buscas na internet utilizando Tavily."""

    try:
        # Inicializar Tavily
        tavily_search = TavilySearch(
            max_results=3,
            search_depth="advanced"
        )

        # Realizar pesquisa
        search_results = tavily_search.invoke({
            "query": state["query"]
        })

        # Extrair resultados
        if isinstance(search_results, dict):
            results = search_results.get("results", [])
        elif isinstance(search_results, list):
            results = search_results
        else:
            results = [search_results]

        # Formatar resultados
        formatted_results = "\n".join(
            f"{r.get('title', '')}: {r.get('content', '')}"
            if isinstance(r, dict) else str(r)
            for r in results
        )

        state["search_results"] = (
            formatted_results or "Nenhum resultado encontrado."
        )

    except Exception as e:
        print(f"Erro no Tavily: {e}")
        raise

    return state


In [7]:

from agentops.sdk.decorators import operation
from langchain_core.prompts import ChatPromptTemplate

# Registrar operação no AgentOps
@operation(name="summarize_results")
def summarize_results(state: AgentState) -> AgentState:
    """Resume os resultados da pesquisa utilizando LLM."""

    # Criar prompt
    prompt = ChatPromptTemplate.from_template(
        "Resuma as seguintes informações em um parágrafo conciso em português.\n\nInformações: {search_results}"
        
    )

    # Criar cadeia de processamento
    chain = prompt | model

    # Executar modelo
    summary = chain.invoke({
        "search_results": state["search_results"]
    }).content

    # Atualizar estado
    state["summary"] = summary

    return state


In [8]:
def generate_answer(state: AgentState) -> AgentState:
    prompt = ChatPromptTemplate.from_template(
        "Com base na sua pergunta '{query}' e nas informações resumidas abaixo, forneça uma resposta detalhada em português, utilizando bullet points quando necessário.\n\nResumo: {summary}"
    )
    chain = prompt | model
    final_answer = chain.invoke({"query": state["query"], "summary": state["summary"]}).content
    state["final_answer"] = final_answer
    return state

In [9]:
from langgraph.graph import StateGraph, START, END

# Construir workflow
workflow = StateGraph(AgentState)

workflow.add_node("web_search", web_search)
workflow.add_node("summarize_results", summarize_results)
workflow.add_node("generate_answer", generate_answer)

workflow.add_edge(START, "web_search")
workflow.add_edge("web_search", "summarize_results")
workflow.add_edge("summarize_results", "generate_answer")
workflow.add_edge("generate_answer", END)

graph = workflow.compile()

In [12]:
from agentops.sdk.decorators import trace, operation, tool
@trace(name="langgraph-workflow")
def execute_workflow(query: str) -> str:
    initial_state = {
        "query": query,
        "search_results": "",
        "summary": "",
        "final_answer": ""
    }

    final_state = graph.invoke(initial_state)

    answer = final_state.get("final_answer")

    if not answer:
        raise RuntimeError(
            "O nó generate_answer não produziu uma resposta final."
        )

    return answer

In [13]:

# Exemplo de consulta
query = "Quais são os cursos da scoras academy?"

# Executar fluxo de trabalho
answer = execute_workflow(query)

# Exibir resposta final
print("Resposta Final:\n")
print(answer)

🖇 AgentOps: Session Replay for langgraph-workflow trace: https://app.agentops.ai/sessions?trace_id=37df4348005c843150a483c4c8272f47
🖇 AgentOps: Session Replay for langgraph-workflow.session trace: https://app.agentops.ai/sessions?trace_id=37df4348005c843150a483c4c8272f47


Resposta Final:

A Scoras Academy, fundada por Anderson Amaral, é uma instituição de ensino focada em inteligência artificial e desenvolvimento de software, oferecendo uma variedade de cursos projetados para capacitar tanto iniciantes quanto profissionais experientes. Aqui estão os principais aspectos dos cursos oferecidos pela Scoras Academy:

- **Fundamentos de LLMs (Modelos de Linguagem de Grande Escala):**
  - Introdução aos conceitos básicos de LLMs.
  - Aplicações práticas e casos de uso.

- **RAG (Retrieval-Augmented Generation):**
  - Técnicas para melhorar a geração de texto com recuperação de informações.
  - Implementação prática em projetos reais.

- **Configuração do Ambiente de Desenvolvimento:**
  - Guia passo a passo para configurar um ambiente de desenvolvimento eficiente.
  - Uso de Python devido à sua simplicidade e eficiência.

- **Conceitos Avançados em IA:**
  - Exploração de técnicas avançadas em inteligência artificial.
  - Aplicação prática desses conceitos em 

In [14]:

# Exemplo de consulta
query = "Quais são os cursos da scoras academy?"

# Executar fluxo de trabalho
answer = execute_workflow(query)

# Exibir resposta final
print("Resposta Final:\n")
print(answer)

🖇 AgentOps: Session Replay for langgraph-workflow trace: https://app.agentops.ai/sessions?trace_id=d07704966213d44cfe1ba2febd9477a4
🖇 AgentOps: Session Replay for langgraph-workflow.session trace: https://app.agentops.ai/sessions?trace_id=d07704966213d44cfe1ba2febd9477a4


Resposta Final:

A Scoras Academy, fundada por Anderson Amaral, oferece uma variedade de cursos focados em inteligência artificial e machine learning, com ênfase na aplicação prática. Aqui estão os principais aspectos dos cursos oferecidos:

- **Fundamentos de LLMs (Modelos de Linguagem de Grande Escala)**:
  - Introdução aos conceitos básicos e avançados de LLMs.
  - Aplicações práticas e casos de uso.

- **RAG (Retrieval-Augmented Generation)**:
  - Técnicas para melhorar a geração de texto com recuperação de informações.
  - Implementação prática em projetos reais.

- **Foco na Aplicação Prática**:
  - Os cursos são projetados para transformar teoria em prática.
  - Projetos prontos para adaptação, cobrindo uma ampla gama de aplicações em IA.

- **Público-Alvo**:
  - Adequado tanto para iniciantes quanto para profissionais experientes.
  - Conteúdo exclusivo e aprendizado prático para todos os níveis de conhecimento.

- **Recursos e Suporte**:
  - Acesso a scripts detalhados e mater